# RAG Pipeline Test
Dieses Notebook testet alle Komponenten der RAG-Pipeline Schritt für Schritt.

## 0. Setup — Pfade & Imports

In [1]:
import sys
sys.path.append('../')  # damit Python die Module im Hauptordner findet

from dotenv import load_dotenv
load_dotenv('../.env')  # API-Keys laden

True

## 1. Loader testen

In [2]:
from rag.loader import DocumentLoader

# Test-PDF laden (liegt im selben Ordner wie dieses Notebook)
loader = DocumentLoader('machine_learning_intro.pdf')
documents = loader.load()

print(f'Anzahl Seiten: {len(documents)}')
print(f'Erste Seite (erste 300 Zeichen):')
print(documents[0].page_content[:300])
print(f'Metadaten: {documents[0].metadata}')

Anzahl Seiten: 3
Erste Seite (erste 300 Zeichen):
Introduction to Machine Learning
Chapter 1: What is Machine Learning?
Machine Learning (ML) is a subset of Artificial Intelligence (AI) that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs that can
access data and u
Metadaten: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-03-06T17:28:33+00:00', 'source': 'machine_learning_intro.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


## 2. Chunker testen

In [4]:
from rag.chunking import DocumentChunker

chunker = DocumentChunker(chunk_size=1000, chunk_overlap=200)
chunks = chunker.chunk(documents)

print(f'Anzahl Chunks: {len(chunks)}')
print(f'Erster Chunk (erste 300 Zeichen):')
print(chunks[0].page_content[:300])
print(f'Metadaten: {chunks[0].metadata}')

Anzahl Chunks: 7
Erster Chunk (erste 300 Zeichen):
Introduction to Machine Learning
Chapter 1: What is Machine Learning?
Machine Learning (ML) is a subset of Artificial Intelligence (AI) that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs that can
access data and u
Metadaten: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-03-06T17:28:33+00:00', 'source': 'machine_learning_intro.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


## 3. Embeddings testen

In [5]:
from rag.embeddings import EmbeddingGenerator

embedding_gen = EmbeddingGenerator()
embedding_model = embedding_gen.get_embedding_model()

# Einen Test-Vektor generieren
test_vector = embedding_model.embed_query('Was ist maschinelles Lernen?')

print(f'Vektor-Dimension: {len(test_vector)}')
print(f'Erste 5 Werte: {test_vector[:5]}')

Vektor-Dimension: 3072
Erste 5 Werte: [-0.024391094, 0.011332491, 0.0010179622, -0.062313493, -0.015748147]


## 4. VectorStore testen

In [ ]:
from rag.vectorstore import VectorStore

# VectorStore initialisieren — Speicherort explizit auf Projektebene setzen
vs = VectorStore(persist_directory='../vector_store')

# Chunks speichern
vs.store(chunks)
print('Chunks erfolgreich gespeichert!')

# Retriever holen und testen
retriever = vs.get_retriever(k=3)
results = retriever.invoke('What is machine learning?')

print(f'Gefundene Chunks: {len(results)}')
for i, doc in enumerate(results):
    print(f'\n--- Chunk {i+1} (Seite {doc.metadata.get("page", 0) + 1}) ---')
    print(doc.page_content[:200])

## 5. RAG Chain testen

In [ ]:
from rag.chain import RAGChain

# Mit Groq testen
chain = RAGChain(retriever=retriever, llm='groq')
result = chain.run('Was ist maschinelles Lernen?')

print('=== ANTWORT ===')
print(result['answer'])

print('\n=== QUELLEN ===')
for doc in result['sources']:
    print(f"- Seite {doc.metadata.get('page', 0) + 1}: {doc.page_content[:100]}...")

In [ ]:
# Mit Gemini testen
chain_gemini = RAGChain(retriever=retriever, llm='gemini')
result_gemini = chain_gemini.run('Was ist maschinelles Lernen?')

print('=== ANTWORT (Gemini) ===')
print(result_gemini['answer'])